# STRAT-002 v6: Realized Exit Comparison

Comparing:
- v5: Simple 30% trail (current best)
- v6: MVRV context + SOPR realized exit + trail

Entry: SOPR < 1 AND STH-SOPR < 1 AND Realized Loss Z > 0.5

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from numba import njit
import warnings
warnings.filterwarnings('ignore')

print("STRAT-002 v6: Realized Exit Comparison 🎯")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(mvrv, how='inner').join(realized_loss, how='inner')
df = df.sort_index()
df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()
df['sopr_excess'] = df['sopr'] - 1
df['sopr_excess_z14'] = (df['sopr_excess'] - df['sopr_excess'].rolling(14).mean()) / df['sopr_excess'].rolling(14).std()
df = df[df.index >= '2019-01-01'].dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

In [ ]:
# STRAT-002 Entry: SOPR < 1 AND STH-SOPR < 1 AND RL Z > 0.5
entry_cond = (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['rl_zscore'] > 0.5)
entries = entry_cond & ~entry_cond.shift(1).fillna(False)

print(f"STRAT-002 Entry signals: {entries.sum()}")
print(f"Entry dates:")
for d in df[entries].index:
    print(f"  {d.date()}: ${df.loc[d, 'price']:,.0f}")

In [ ]:
@njit
def exit_simple_trail(price_arr, sopr_arr, sopr_z_arr, mvrv_arr, entry_idx, trail_pct=0.30):
    """v5: Simple trailing stop"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        if price > peak:
            peak = price
        if price <= peak * (1 - trail_pct):
            return j, price, 'trail'
    return len(price_arr) - 1, price_arr[-1], 'hold'


@njit
def exit_realized_v6(price_arr, sopr_arr, sopr_z_arr, mvrv_arr, entry_idx,
                     mvrv_context=2.0, sopr_exit=1.05, sopr_z_exit=1.5,
                     trail_before=0.30, trail_after=0.20):
    """
    v6: Realized profit exit
    - Before trigger: wide trail (30%)
    - After MVRV + SOPR trigger: tighter trail (20%)
    """
    entry_price = price_arr[entry_idx]
    peak = entry_price
    realized_triggered = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        sopr = sopr_arr[j]
        sopr_z = sopr_z_arr[j]
        mvrv = mvrv_arr[j]
        
        if price > peak:
            peak = price
        
        # Check for realized exit trigger
        if not realized_triggered:
            if mvrv > mvrv_context and (sopr > sopr_exit or sopr_z > sopr_z_exit):
                realized_triggered = True
        
        # Use appropriate trail
        trail = trail_after if realized_triggered else trail_before
        
        if price <= peak * (1 - trail):
            reason = 'realized_trail' if realized_triggered else 'trail'
            return j, price, reason
    
    return len(price_arr) - 1, price_arr[-1], 'hold'

In [ ]:
def run_backtest(df, entries, exit_func, initial_capital=100000, **kwargs):
    """Run backtest"""
    price_arr = df['price'].values
    sopr_arr = df['sopr'].values
    sopr_z_arr = df['sopr_excess_z14'].values
    mvrv_arr = df['mvrv'].values
    dates = df.index
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        exit_idx, exit_price, exit_reason = exit_func(price_arr, sopr_arr, sopr_z_arr, mvrv_arr, entry_idx, **kwargs)
        
        entry_price = price_arr[entry_idx]
        peak_price = max(price_arr[entry_idx:exit_idx+1])
        net_return = (exit_price / entry_price) - 1 - 0.002
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'peak_price': peak_price,
            'exit_price': exit_price,
            'exit_reason': exit_reason,
            'entry_mvrv': mvrv_arr[entry_idx],
            'exit_mvrv': mvrv_arr[exit_idx],
            'exit_sopr': sopr_arr[exit_idx],
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    return trades_df


def calc_metrics(trades, initial_capital=100000):
    if len(trades) == 0:
        return None
    final = trades['equity'].iloc[-1]
    total_ret = (final / initial_capital) - 1
    years = (trades['exit_date'].iloc[-1] - trades['entry_date'].iloc[0]).days / 365.25
    cagr = (1 + total_ret) ** (1 / years) - 1 if years > 0 else 0
    win_rate = (trades['net_return'] > 0).mean()
    returns = trades['net_return'].values
    sharpe = (returns.mean() / returns.std()) * np.sqrt(len(trades)/years) if returns.std() > 0 and years > 0 else 0
    
    winners = returns[returns > 0]
    losers = returns[returns <= 0]
    profit_factor = abs(winners.sum() / losers.sum()) if len(losers) > 0 and losers.sum() != 0 else np.inf
    
    equity = [initial_capital] + list(trades['equity'])
    peak, max_dd = equity[0], 0
    for eq in equity:
        if eq > peak: peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd: max_dd = dd
    
    return {
        'total_return': total_ret,
        'cagr': cagr,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'win_rate': win_rate,
        'profit_factor': profit_factor,
        'n_trades': len(trades),
        'avg_hold': trades['days_held'].mean(),
        'final_equity': final
    }

---
## Head-to-Head Comparison

In [ ]:
print("STRAT-002: v5 vs v6 COMPARISON")
print("Entry: SOPR < 1 AND STH-SOPR < 1 AND RL Z > 0.5")
print("="*100)

strategies = [
    # v5 variants
    ('v5: Simple 30% Trail', exit_simple_trail, {'trail_pct': 0.30}),
    ('v5: Simple 25% Trail', exit_simple_trail, {'trail_pct': 0.25}),
    ('v5: Simple 20% Trail', exit_simple_trail, {'trail_pct': 0.20}),
    
    # v6 variants - Realized Exit
    ('v6: MVRV>2.0 + SOPR>1.05 → 30/20', exit_realized_v6,
     {'mvrv_context': 2.0, 'sopr_exit': 1.05, 'sopr_z_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.20}),
    ('v6: MVRV>2.0 + SOPR>1.05 → 30/15', exit_realized_v6,
     {'mvrv_context': 2.0, 'sopr_exit': 1.05, 'sopr_z_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('v6: MVRV>2.5 + SOPR>1.05 → 30/20', exit_realized_v6,
     {'mvrv_context': 2.5, 'sopr_exit': 1.05, 'sopr_z_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.20}),
    ('v6: MVRV>2.5 + SOPR>1.05 → 30/15', exit_realized_v6,
     {'mvrv_context': 2.5, 'sopr_exit': 1.05, 'sopr_z_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('v6: MVRV>2.0 + SOPR Z>1.5 → 30/20', exit_realized_v6,
     {'mvrv_context': 2.0, 'sopr_exit': 99, 'sopr_z_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.20}),
    ('v6: MVRV>2.0 + SOPR Z>2.0 → 30/15', exit_realized_v6,
     {'mvrv_context': 2.0, 'sopr_exit': 99, 'sopr_z_exit': 2.0, 'trail_before': 0.30, 'trail_after': 0.15}),
]

print(f"{'Strategy':<40} {'Return':>12} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'Win%':>7} {'PF':>6} {'Trades':>7}")
print("-"*100)

results = []
for name, func, kwargs in strategies:
    trades = run_backtest(df, entries, func, **kwargs)
    m = calc_metrics(trades)
    if m:
        pf = f"{m['profit_factor']:.1f}" if m['profit_factor'] < 100 else "∞"
        print(f"{name:<40} {m['total_return']*100:>+11.0f}% {m['cagr']*100:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']*100:>7.0f}% {m['win_rate']*100:>6.0f}% {pf:>6} {m['n_trades']:>7}")
        results.append({'name': name, 'metrics': m, 'trades': trades})

In [ ]:
# Find best v5 and v6
best_v5 = max([r for r in results if 'v5' in r['name']], key=lambda x: x['metrics']['total_return'])
best_v6 = max([r for r in results if 'v6' in r['name']], key=lambda x: x['metrics']['total_return'])

print("\n" + "="*80)
print("BEST v5 vs BEST v6")
print("="*80)

print(f"\n{'Metric':<20} {'v5 (Simple Trail)':>25} {'v6 (Realized Exit)':>25} {'Diff':>15}")
print("-"*85)

m5, m6 = best_v5['metrics'], best_v6['metrics']
metrics = [
    ('Total Return', 'total_return', '%', 100),
    ('CAGR', 'cagr', '%', 100),
    ('Sharpe', 'sharpe', '', 1),
    ('Max Drawdown', 'max_dd', '%', 100),
    ('Win Rate', 'win_rate', '%', 100),
    ('Profit Factor', 'profit_factor', '', 1),
    ('Trades', 'n_trades', '', 1),
    ('Avg Hold (days)', 'avg_hold', 'd', 1),
    ('Final Equity', 'final_equity', '$', 1),
]

for label, key, suffix, mult in metrics:
    v5_val = m5[key] * mult
    v6_val = m6[key] * mult
    diff = v6_val - v5_val
    
    if key == 'final_equity':
        print(f"{label:<20} ${v5_val:>24,.0f} ${v6_val:>24,.0f} ${diff:>+14,.0f}")
    elif suffix == '%':
        print(f"{label:<20} {v5_val:>24.1f}% {v6_val:>24.1f}% {diff:>+14.1f}%")
    elif suffix == 'd':
        print(f"{label:<20} {v5_val:>24.0f}d {v6_val:>24.0f}d {diff:>+14.0f}d")
    else:
        print(f"{label:<20} {v5_val:>25.2f} {v6_val:>25.2f} {diff:>+15.2f}")

print(f"\nv5: {best_v5['name']}")
print(f"v6: {best_v6['name']}")
print(f"\n🏆 WINNER: {'v6 (Realized Exit)' if m6['total_return'] > m5['total_return'] else 'v5 (Simple Trail)'}")

In [ ]:
# Trade-by-trade comparison
print("\n" + "="*120)
print("TRADE-BY-TRADE COMPARISON")
print("="*120)

t5 = best_v5['trades']
t6 = best_v6['trades']

print(f"\n{'#':<3} {'Entry':<12} {'v5 Exit':<12} {'v5 Return':>12} {'v6 Exit':<12} {'v6 Return':>12} {'v6 Reason':<15} {'Better':>10}")
print("-"*100)

for i in range(max(len(t5), len(t6))):
    if i < len(t5) and i < len(t6):
        r5 = t5.iloc[i]
        r6 = t6.iloc[i]
        better = 'v6 ✅' if r6['net_return'] > r5['net_return'] else 'v5 ✅'
        print(f"{i+1:<3} {r5['entry_date'].strftime('%Y-%m-%d'):<12} {r5['exit_date'].strftime('%Y-%m-%d'):<12} {r5['net_return']*100:>+11.0f}% {r6['exit_date'].strftime('%Y-%m-%d'):<12} {r6['net_return']*100:>+11.0f}% {r6['exit_reason']:<15} {better:>10}")

In [ ]:
# Detailed v6 trade log
print("\n" + "="*120)
print(f"v6 DETAILED TRADE LOG: {best_v6['name']}")
print("="*120)

t6 = best_v6['trades']
print(f"\n{'Entry':<12} {'Exit':<12} {'Days':>6} {'Entry $':>10} {'Peak $':>10} {'Exit $':>10} {'Return':>10} {'Reason':<15} {'Exit MVRV':>10}")
print("-"*110)

for _, t in t6.iterrows():
    print(f"{t['entry_date'].strftime('%Y-%m-%d'):<12} {t['exit_date'].strftime('%Y-%m-%d'):<12} {t['days_held']:>6} ${t['entry_price']:>9,.0f} ${t['peak_price']:>9,.0f} ${t['exit_price']:>9,.0f} {t['net_return']*100:>+9.0f}% {t['exit_reason']:<15} {t['exit_mvrv']:>10.2f}")

In [ ]:
# Plot equity curves
import plotly.graph_objects as go

fig = go.Figure()

# v5
t5 = best_v5['trades']
eq5_dates = [t5['entry_date'].iloc[0]] + list(t5['exit_date'])
eq5_vals = [100000] + list(t5['equity'])
fig.add_trace(go.Scatter(x=eq5_dates, y=eq5_vals, name=f"v5: {best_v5['name']}", line=dict(color='blue')))

# v6
t6 = best_v6['trades']
eq6_dates = [t6['entry_date'].iloc[0]] + list(t6['exit_date'])
eq6_vals = [100000] + list(t6['equity'])
fig.add_trace(go.Scatter(x=eq6_dates, y=eq6_vals, name=f"v6: {best_v6['name']}", line=dict(color='green')))

# B&H
bh = 100000 * (df['price'] / df['price'].iloc[0])
fig.add_trace(go.Scatter(x=bh.index, y=bh.values, name='Buy & Hold', line=dict(color='gray', dash='dash')))

fig.update_layout(
    title='STRAT-002: v5 (Simple Trail) vs v6 (Realized Exit)',
    yaxis_title='Equity ($)',
    yaxis_type='log',
    height=500
)
fig.show()

---
## Summary

In [ ]:
print("\n" + "="*70)
print("STRAT-002 VERSION COMPARISON SUMMARY")
print("="*70)

print(f"""
📊 ENTRY (Both versions):
   SOPR < 1 AND STH-SOPR < 1 AND Realized Loss Z > 0.5
   (Capitulation direction + intensity)

📈 v5 EXIT (Simple Trail):
   30% trailing stop from peak (always active)
   Return: {best_v5['metrics']['total_return']*100:+,.0f}%
   Sharpe: {best_v5['metrics']['sharpe']:.2f}

📈 v6 EXIT (Realized Profit):
   Before trigger: 30% trail
   After MVRV > X + SOPR > 1.05: tighten to 15-20% trail
   Return: {best_v6['metrics']['total_return']*100:+,.0f}%
   Sharpe: {best_v6['metrics']['sharpe']:.2f}

🏆 WINNER: {'v6 (Realized Exit)' if m6['total_return'] > m5['total_return'] else 'v5 (Simple Trail)'}
   Difference: {(m6['total_return'] - m5['total_return'])*100:+,.0f}% return
""")

print("💡 KEY INSIGHT:")
print("   - Entry uses REALIZED metrics (SOPR < 1 = selling at loss)")
print("   - Exit uses REALIZED metrics (SOPR > 1.05 = selling at profit)")
print("   - Unrealized (MVRV) provides CONTEXT, not trigger")
print("   - Action beats State for both entries AND exits!")